In [18]:
import pandas as pd
from pathlib import Path
import gspread
from gspread_dataframe import set_with_dataframe
from google.oauth2.service_account import Credentials
from gspread_formatting import *

In [19]:
DIR_PATH = Path("Consolidated_Schedule_of_Investments/2026-03-31/")
CONSOLIDATED_CSV_PATH = Path(DIR_PATH, "outputs/ck0001902649-20260331.csv")
ANALYSIS_PATH = Path(DIR_PATH, "analysis")
ANALYSIS_PATH.mkdir(exist_ok=True)

DIR_PATH.exists(), CONSOLIDATED_CSV_PATH.exists(), ANALYSIS_PATH.exists()

(True, True, True)

In [20]:
df_investment = pd.read_csv(CONSOLIDATED_CSV_PATH)
df_investment.info()

<class 'pandas.DataFrame'>
RangeIndex: 469 entries, 0 to 468
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   issuer                 469 non-null    str    
 1   instrument             469 non-null    str    
 2   industry               469 non-null    str    
 3   ref                    466 non-null    str    
 4   floor                  291 non-null    float64
 5   investment_type        469 non-null    str    
 6   spread                 466 non-null    str    
 7   total_coupon           466 non-null    float64
 8   maturity               466 non-null    str    
 9   principal              383 non-null    str    
 10  cost                   469 non-null    str    
 11  fair_value             469 non-null    str    
 12  total_cash_investment  374 non-null    float64
 13  notes                  307 non-null    str    
 14  part                   469 non-null    str    
 15  table_index      

In [21]:
df_investment[df_investment["issuer"].str.contains("Skydio, Inc", na=False)]["cost"]

8     7709786
9     3722419
10    -27,581
Name: cost, dtype: str

In [22]:
df_investment.head(275)

,issuer,instrument,industry,ref,floor,investment_type,spread,total_coupon,maturity,principal,cost,fair_value,total_cash_investment,notes,part,table_index,row_index
0,"Arcline FM Holdings, LLC (Fairbanks Morse)",First Lien Term Loan,aerospace and defense,SOFR(Q),0.75,debt investments(a),2.75,6.45,6/23/2030,2875972,2881744,2883967,0.11,NaN,ck0001902649-20260331,0,4
1,Bleriot US Bidco Inc.,First Lien Term Loan,aerospace and defense,SOFR(Q),NaN,debt investments(a),2.50,6.20,10/30/2026,3289161,3289123,3297960,0.13,NaN,ck0001902649-20260331,0,5
2,Cobham Ultra US Co-Borrower LLC (Ultra Electro...,First Lien Term Loan,aerospace and defense,SOFR(S),NaN,debt investments(a),4.18,7.79,8/4/2029,2416311,2417686,2425070,0.09,NaN,ck0001902649-20260331,0,6
3,Chromalloy Holdings LLC,First Lien Term Loan,aerospace and defense,SOFR(Q),NaN,debt investments(a),3.25,6.91,3/31/2031,3792109,3806939,3799523,0.13,NaN,ck0001902649-20260331,0,7
4,"Engineering Research Holding LLC (Astrion, Inc.)",First Lien Term Loan,aerospace and defense,SOFR(Q),1.00,debt investments(a),5.00,8.70,8/29/2031,22749759,22484214,18048180,0.69,NaN,ck0001902649-20260331,0,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
270,Spartan Bidco Pty Ltd (StarRez) (Australia),First Lien Revolver,internet software and services,SOFR(Q),0.75,debt investments(a),6.65,10.32,1/24/2028,187753,181798,181895,0.01,C/E,ck0001902649-20260331,8,6
271,"Anthracite Buyer, Inc. (Coalfire)",First Lien Term Loan,it services,SOFR(Q),0.75,debt investments(a),4.50,8.15,12/3/2032,17151957,17070188,16980437,0.64,E,ck0001902649-20260331,8,9
272,"Anthracite Buyer, Inc. (Coalfire)",First Lien Revolver,it services,SOFR(Q),0.75,debt investments(a),4.50,8.15,12/3/2032,NaN,"-20,442","-42,880",NaN,D/E,ck0001902649-20260331,8,10
273,"Avalara, Inc.",First Lien Term Loan,it services,SOFR(Q),NaN,debt investments(a),2.75,6.45,3/26/2032,461518,459542,451711,0.02,NaN,ck0001902649-20260331,8,11


In [23]:
# convert cost 
for col in ["cost", "fair_value"]:
    df_investment[col] = (
        df_investment[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace(r"^\((.*)\)$", r"-\1", regex=True)
        .replace("nan", pd.NA)
    )

    df_investment[col] = pd.to_numeric(df_investment[col], errors="coerce").astype("Int64")

df_investment.drop(columns=["part", "table_index", "row_index"], inplace=True)
df_investment

,issuer,instrument,industry,ref,floor,investment_type,spread,total_coupon,maturity,principal,cost,fair_value,total_cash_investment,notes
0,"Arcline FM Holdings, LLC (Fairbanks Morse)",First Lien Term Loan,aerospace and defense,SOFR(Q),0.75,debt investments(a),2.75,6.45,6/23/2030,2875972,2881744,2883967,0.11,NaN
1,Bleriot US Bidco Inc.,First Lien Term Loan,aerospace and defense,SOFR(Q),NaN,debt investments(a),2.50,6.20,10/30/2026,3289161,3289123,3297960,0.13,NaN
2,Cobham Ultra US Co-Borrower LLC (Ultra Electro...,First Lien Term Loan,aerospace and defense,SOFR(S),NaN,debt investments(a),4.18,7.79,8/4/2029,2416311,2417686,2425070,0.09,NaN
3,Chromalloy Holdings LLC,First Lien Term Loan,aerospace and defense,SOFR(Q),NaN,debt investments(a),3.25,6.91,3/31/2031,3792109,3806939,3799523,0.13,NaN
4,"Engineering Research Holding LLC (Astrion, Inc.)",First Lien Term Loan,aerospace and defense,SOFR(Q),1.00,debt investments(a),5.00,8.70,8/29/2031,22749759,22484214,18048180,0.69,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
464,"KKR Apple Bidco, LLC",First Lien Term Loan,transportation infrastructure,SOFR(M),NaN,debt investments(a),2.50,6.17,9/23/2031,330359,330359,330927,0.01,NaN
465,"OpenMarket, Inc. (Infobip) (United Kingdom)",First Lien Term Loan,wireless telecommunication services,SOFR(Q),0.75,debt investments(a),5.50,9.20,6/11/2029,43689841,43166729,42992725,1.63,NaN
466,Streamland Media Holdings LLC,Common Stock,media,NaN,NaN,equity securities,NaN,NaN,NaN,12363,645215,0,NaN,NaN
467,"48forty Intermediate Holdings, Inc.",Class A-1 Common Units,paper and forest products,NaN,NaN,equity securities,NaN,NaN,NaN,285,0,0,NaN,E/I/J


In [24]:
df_investment["profit"] = df_investment["fair_value"] - df_investment["cost"]
df_investment["is_profit"] = df_investment["profit"] > 0
df_investment["is_loss"] = df_investment["profit"] < 0
df_investment["is_breakeven"] = df_investment["profit"] == 0
df_investment['Profit_Loss'] = df_investment['fair_value'] - df_investment['cost']
df_investment

,issuer,instrument,industry,ref,floor,investment_type,spread,total_coupon,maturity,principal,cost,fair_value,total_cash_investment,notes,profit,is_profit,is_loss,is_breakeven,Profit_Loss
0,"Arcline FM Holdings, LLC (Fairbanks Morse)",First Lien Term Loan,aerospace and defense,SOFR(Q),0.75,debt investments(a),2.75,6.45,6/23/2030,2875972,2881744,2883967,0.11,NaN,2223,True,False,False,2223
1,Bleriot US Bidco Inc.,First Lien Term Loan,aerospace and defense,SOFR(Q),NaN,debt investments(a),2.50,6.20,10/30/2026,3289161,3289123,3297960,0.13,NaN,8837,True,False,False,8837
2,Cobham Ultra US Co-Borrower LLC (Ultra Electro...,First Lien Term Loan,aerospace and defense,SOFR(S),NaN,debt investments(a),4.18,7.79,8/4/2029,2416311,2417686,2425070,0.09,NaN,7384,True,False,False,7384
3,Chromalloy Holdings LLC,First Lien Term Loan,aerospace and defense,SOFR(Q),NaN,debt investments(a),3.25,6.91,3/31/2031,3792109,3806939,3799523,0.13,NaN,-7416,False,True,False,-7416
4,"Engineering Research Holding LLC (Astrion, Inc.)",First Lien Term Loan,aerospace and defense,SOFR(Q),1.00,debt investments(a),5.00,8.70,8/29/2031,22749759,22484214,18048180,0.69,NaN,-4436034,False,True,False,-4436034
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
464,"KKR Apple Bidco, LLC",First Lien Term Loan,transportation infrastructure,SOFR(M),NaN,debt investments(a),2.50,6.17,9/23/2031,330359,330359,330927,0.01,NaN,568,True,False,False,568
465,"OpenMarket, Inc. (Infobip) (United Kingdom)",First Lien Term Loan,wireless telecommunication services,SOFR(Q),0.75,debt investments(a),5.50,9.20,6/11/2029,43689841,43166729,42992725,1.63,NaN,-174004,False,True,False,-174004
466,Streamland Media Holdings LLC,Common Stock,media,NaN,NaN,equity securities,NaN,NaN,NaN,12363,645215,0,NaN,NaN,-645215,False,True,False,-645215
467,"48forty Intermediate Holdings, Inc.",Class A-1 Common Units,paper and forest products,NaN,NaN,equity securities,NaN,NaN,NaN,285,0,0,NaN,E/I/J,0,False,False,True,0


In [25]:
df_summary = pd.DataFrame({
    "a": [
        "Total Investment Count",
        "Total Cost", 
        "Total FV", 
        "P&L"
    ],
    "b": [
        len(df_investment),
        int(df_investment["cost"].sum()),
        int(df_investment["fair_value"].sum()),
        int(df_investment["fair_value"].sum())-int(df_investment["cost"].sum()),
    ]
})

df_summary

,a,b
0,Total Investment Count,469
1,Total Cost,2391096696
2,Total FV,2354502240
3,P&L,-36594456


In [26]:
# instrument-Level Analysis
df_instrument_level_analysis = (
    df_investment.groupby("instrument")
      .agg(
          total_investments=("issuer", "count"),

          # counts
          profitable_count=("is_profit", "sum"),
          loss_count=("is_loss", "sum"),
          breakeven_count=("is_breakeven", "sum"),

          # financials
          total_cost=("cost", "sum"),
          total_fair_value=("fair_value", "sum"),
          total_profit=("profit", "sum"),

          # averages
          avg_profit=("profit", "mean"),

          # risk
          max_profit=("profit", "max"),
          max_loss=("profit", "min")
      )
)

df_instrument_level_analysis.reset_index(names="instrument", inplace=True)
df_instrument_level_analysis["avg_profit"] = round(df_instrument_level_analysis["avg_profit"], 2)

# win, breakeven, and loss rate
df_instrument_level_analysis["win_rate"] = round(
    df_instrument_level_analysis["profitable_count"] /
    df_instrument_level_analysis["total_investments"]
, 2)

df_instrument_level_analysis["breakeven_rate"] = round(
    df_instrument_level_analysis["breakeven_count"] /
    df_instrument_level_analysis["total_investments"]
, 2)

df_instrument_level_analysis["loss_rate"] = round(
    df_instrument_level_analysis["loss_count"] /
    df_instrument_level_analysis["total_investments"]
, 2)

# sort by total no investments
df_instrument_level_analysis = df_instrument_level_analysis.sort_values(
    by="total_fair_value", ascending=False
)


column_mapping = {
    "instrument": "Instrument",
    "total_investments": "Total Investments Count",
    "total_cost": "Total Instrument Cost",
    "total_fair_value": "Total Instrument FV",
    "total_profit": "Total Instrument P&L",
    "avg_profit": "Avg Profit/Instrument",
    "max_profit": "Max Profit Instrument",
    "max_loss": "Max Loss Instrument",
    "profitable_count": "Profitable Instrument Count",
    "win_rate": "Profitable %",
    "breakeven_count": "Breakeven Instrument Count",
    "breakeven_rate": "Breakeven %",
    "loss_count": "Loss Instrument Count",
    "loss_rate": "Loss %"
}
desired_order = [
    "Instrument",
    "Total Investments Count",
    "Total Instrument Cost",
    "Total Instrument FV",
    "Total Instrument P&L",
    "Avg Profit/Instrument",
    "Max Profit Instrument",
    "Max Loss Instrument",
    "Profitable Instrument Count",
    "Profitable %",
    "Breakeven Instrument Count",
    "Breakeven %",
    "Loss Instrument Count",
    "Loss %"
]

df_instrument_level_analysis = df_instrument_level_analysis.rename(columns=column_mapping).reindex(columns=desired_order)

display(df_instrument_level_analysis)

,Instrument,Total Investments Count,Total Instrument Cost,Total Instrument FV,Total Instrument P&L,Avg Profit/Instrument,Max Profit Instrument,Max Loss Instrument,Profitable Instrument Count,Profitable %,Breakeven Instrument Count,Breakeven %,Loss Instrument Count,Loss %
19,First Lien Term Loan,283,2067646155,2036917884,-30728271,-108580.46,641498,-4436034,90,0.32,1,0.0,192,0.68
3,First Lien Delayed Draw Term Loan,88,219968696,218950050,-1018646,-11575.52,437255,-762261,37,0.42,2,0.02,49,0.56
11,First Lien Incremental Term Loan,4,20238865,20171821,-67044,-16761.0,16097,-60713,1,0.25,0,0.0,3,0.75
25,First Lien Tranche B-1 Term Loan,1,16677136,16034226,-642910,-642910.0,-642910,-642910,0,0.0,0,0.0,1,1.0
20,First Lien Term Loan (4.5% Exit Fee),1,9391825,9362401,-29424,-29424.0,-29424,-29424,0,0.0,0,0.0,1,1.0
15,First Lien Revolver,64,8519140,7719958,-799182,-12487.22,64331,-381131,23,0.36,2,0.03,39,0.61
24,First Lien Tranche B Term Loan,1,7752306,7499226,-253080,-253080.0,-253080,-253080,0,0.0,0,0.0,1,1.0
12,First Lien Initial Term Loan,1,6640840,6500539,-140301,-140301.0,-140301,-140301,0,0.0,0,0.0,1,1.0
8,First Lien Delayed Draw Term Loan E,1,5783200,5765984,-17216,-17216.0,-17216,-17216,0,0.0,0,0.0,1,1.0
22,First Lien Term Loan B,3,5268451,5183043,-85408,-28469.33,6688,-84798,1,0.33,0,0.0,2,0.67


In [27]:
# investments_type-Level Analysis
df_investmentsType_level_analysis = (
    df_investment.groupby("investment_type")
      .agg(
          total_investments=("investment_type", "count"),

          # counts
          profitable_count=("is_profit", "sum"),
          loss_count=("is_loss", "sum"),
          breakeven_count=("is_breakeven", "sum"),

          # financials
          total_cost=("cost", "sum"),
          total_fair_value=("fair_value", "sum"),
          total_profit=("profit", "sum"),

          # averages
          avg_profit=("profit", "mean"),

          # risk
          max_profit=("profit", "max"),
          max_loss=("profit", "min")
      )
)

df_investmentsType_level_analysis.reset_index(names="investment_type", inplace=True)
df_investmentsType_level_analysis["avg_profit"] = round(df_investmentsType_level_analysis["avg_profit"], 2)

# win, breakeven, and loss rate
df_investmentsType_level_analysis["win_rate"] = round(
    df_investmentsType_level_analysis["profitable_count"] /
    df_investmentsType_level_analysis["total_investments"]
, 2)

df_investmentsType_level_analysis["breakeven_rate"] = round(
    df_investmentsType_level_analysis["breakeven_count"] /
    df_investmentsType_level_analysis["total_investments"]
, 2)

df_investmentsType_level_analysis["loss_rate"] = round(
    df_investmentsType_level_analysis["loss_count"] /
    df_investmentsType_level_analysis["total_investments"]
, 2)

# sort by total no investments
df_investmentsType_level_analysis = df_investmentsType_level_analysis.sort_values(
    by="total_fair_value", ascending=False
)


column_mapping = {
    "investment_type":"Investment Type",
    "total_investments": "Total Investment Count",
    "total_cost": "Total Investment Type Cost",
    "total_fair_value": "Total Investment Type FV",
    "total_profit": "Total Investment Type P&L",
    "avg_profit": "Avg Profit/Investment Type",
    "max_profit": "Max Profit Investment Type",
    "max_loss": "Max Loss Investment Type",
    "profitable_count": "Profitable Investment Type Count",
    "win_rate": "Profitable %",
    "breakeven_count": "Breakeven Investment Type Count",
    "breakeven_rate": "Breakeven %",
    "loss_count": "Loss Investment Type Count",
    "loss_rate": "Loss %"
}
desired_order = [
    "Investment Type",
    "Total Investment Count",
    "Total Investment Type Cost",
    "Total Investment Type FV",
    "Total Investment Type P&L",
    "Avg Profit/Investment Type",
    "Max Profit Investment Type",
    "Max Loss Investment Type",
    "Profitable Investment Type Count",
    "Profitable %",
    "Breakeven Investment Type Count",
    "Breakeven %",
    "Loss Investment Type Count",
    "Loss %"
]

df_investmentsType_level_analysis = df_investmentsType_level_analysis.rename(columns=column_mapping).reindex(columns=desired_order)

display(df_investmentsType_level_analysis)

,Investment Type,Total Investment Count,Total Investment Type Cost,Total Investment Type FV,Total Investment Type P&L,Avg Profit/Investment Type,Max Profit Investment Type,Max Loss Investment Type,Profitable Investment Type Count,Profitable %,Breakeven Investment Type Count,Breakeven %,Loss Investment Type Count,Loss %
0,debt investments(a),466,2389364013,2353460431,-35903582,-77046.31,641498,-4436034,156,0.33,8,0.02,302,0.65
1,equity securities,3,1732683,1041809,-690874,-230291.33,0,-645215,0,0.0,1,0.33,2,0.67


In [28]:
# Industry-Level Analysis
df_industry_level_analysis = (
    df_investment.groupby("industry")
      .agg(
          total_investments=("issuer", "count"),

          # counts
          profitable_count=("is_profit", "sum"),
          loss_count=("is_loss", "sum"),
          breakeven_count=("is_breakeven", "sum"),

          # financials
          total_cost=("cost", "sum"),
          total_fair_value=("fair_value", "sum"),
          total_profit=("profit", "sum"),

          # averages
          avg_profit=("profit", "mean"),

          # risk
          max_profit=("profit", "max"),
          max_loss=("profit", "min")
      )
)

df_industry_level_analysis.reset_index(names="industry", inplace=True)
df_industry_level_analysis["avg_profit"] = round(df_industry_level_analysis["avg_profit"], 2)


df_industry_level_analysis["win_rate"] = round(
    df_industry_level_analysis["profitable_count"] /
    df_industry_level_analysis["total_investments"]
, 2)

df_industry_level_analysis["breakeven_rate"] = round(
    df_industry_level_analysis["breakeven_count"] /
    df_industry_level_analysis["total_investments"]
, 2)

df_industry_level_analysis["loss_rate"] = round(
    df_industry_level_analysis["loss_count"] /
    df_industry_level_analysis["total_investments"]
, 2)


df_industry_level_analysis = df_industry_level_analysis.sort_values(
    by="total_fair_value", ascending=False
)


column_mapping = {
    "total_investments": "Total Investment Count",
    "total_cost": "Total Investment Cost",
    "total_fair_value": "Total Investment FV",
    "total_profit": "Total Investment P&L",
    "avg_profit": "Avg Profit/Investment",
    "max_profit": "Max Profit Investment",
    "max_loss": "Max Loss Investment",
    "profitable_count": "Profitable Investment Count",
    "win_rate": "Profitable %",
    "breakeven_count": "Breakeven Investment Count",
    "breakeven_rate": "Breakeven %",
    "loss_count": "Loss Investment Count",
    "loss_rate": "Loss %",
    "industry": "Industry"
}
desired_order = [
    "Industry",
    "Total Investment Count",
    "Total Investment Cost",
    "Total Investment FV",
    "Total Investment P&L",
    "Avg Profit/Investment",
    "Max Profit Investment",
    "Max Loss Investment",
    "Profitable Investment Count",
    "Profitable %",
    "Breakeven Investment Count",
    "Breakeven %",
    "Loss Investment Count",
    "Loss %"
]

df_industry_level_analysis = df_industry_level_analysis.rename(columns=column_mapping).reindex(columns=desired_order)

display(df_industry_level_analysis)

,Industry,Total Investment Count,Total Investment Cost,Total Investment FV,Total Investment P&L,Avg Profit/Investment,Max Profit Investment,Max Loss Investment,Profitable Investment Count,Profitable %,Breakeven Investment Count,Breakeven %,Loss Investment Count,Loss %
41,software,81,438259932,429222605,-9037327,-111571.94,317772,-793944,12,0.15,0,0.0,69,0.85
38,professional services,41,242852780,233597811,-9254969,-225730.95,58172,-4246664,10,0.24,0,0.0,31,0.76
9,construction and engineering,35,181604353,184052798,2448445,69955.57,641498,-595929,29,0.83,0,0.0,6,0.17
14,diversified financial services,33,180410108,180736974,326866,9905.03,437255,-105217,8,0.24,1,0.03,24,0.73
5,capital markets,19,129052012,128983687,-68325,-3596.05,467649,-193246,6,0.32,0,0.0,13,0.68
26,insurance,25,116242161,116101374,-140787,-5631.48,71608,-126988,11,0.44,2,0.08,12,0.48
21,health care technology,13,93786474,93015506,-770968,-59305.23,458746,-483468,3,0.23,0,0.0,10,0.77
22,healthcare providers and services,20,89182254,89083537,-98717,-4935.85,184821,-210917,6,0.3,0,0.0,14,0.7
0,aerospace and defense,18,85529495,81191723,-4337772,-240987.33,47397,-4436034,11,0.61,0,0.0,7,0.39
23,"hotels, restaurants and leisure",15,74576620,74378756,-197864,-13190.93,142403,-117809,5,0.33,0,0.0,10,0.67


In [29]:
# asset_list = list(df_asset_class_level_analysis["Asset Class"])
# name = [v.split("—")[0].strip() for v in asset_list]
# perc = [v.split("—")[1] for v in asset_list]

# df_asset_class_level_analysis = df_asset_class_level_analysis.drop(columns=["Asset Class"])
# df_asset_class_level_analysis.insert(0, "Assets", name)
# df_asset_class_level_analysis.insert(1, "percentage", perc)

# df_asset_class_level_analysis

In [30]:
column_mapping = {
    "issuer": "Issuer",
    "instrument": "Instrument",
    "ref": "Ref",
    "floor": "Floor",
    "spread": "Spread",
    "total_coupon": "Total Coupon",
    "maturity": "Maturity",
    "principal": "Principal",
    "cost": "Cost",
    "Profit_Loss": "P&L",
    "fair_value": "Fair Value",
    "investment_type": "Investment Type",
    "industry": "Industry",
    "total_cash_investment": "% of Total Cash and Investment",
    "notes": "Notes",
}


desired_order = [
    "Issuer",
    "Industry",
    "Instrument",
    "Ref",
    "Spread",
    "Total Coupon",
    "Floor",
    "Maturity",
    "Principal",
    "Cost",
    "Fair Value",
    "P&L",
    "Investment Type",
    
    
    
]

df_investment_formatted = df_investment.rename(columns=column_mapping).reindex(columns=desired_order)
df_investment_formatted

,Issuer,Industry,Instrument,Ref,Spread,Total Coupon,Floor,Maturity,Principal,Cost,Fair Value,P&L,Investment Type
0,"Arcline FM Holdings, LLC (Fairbanks Morse)",aerospace and defense,First Lien Term Loan,SOFR(Q),2.75,6.45,0.75,6/23/2030,2875972,2881744,2883967,2223,debt investments(a)
1,Bleriot US Bidco Inc.,aerospace and defense,First Lien Term Loan,SOFR(Q),2.50,6.20,NaN,10/30/2026,3289161,3289123,3297960,8837,debt investments(a)
2,Cobham Ultra US Co-Borrower LLC (Ultra Electro...,aerospace and defense,First Lien Term Loan,SOFR(S),4.18,7.79,NaN,8/4/2029,2416311,2417686,2425070,7384,debt investments(a)
3,Chromalloy Holdings LLC,aerospace and defense,First Lien Term Loan,SOFR(Q),3.25,6.91,NaN,3/31/2031,3792109,3806939,3799523,-7416,debt investments(a)
4,"Engineering Research Holding LLC (Astrion, Inc.)",aerospace and defense,First Lien Term Loan,SOFR(Q),5.00,8.70,1.00,8/29/2031,22749759,22484214,18048180,-4436034,debt investments(a)
...,...,...,...,...,...,...,...,...,...,...,...,...,...
464,"KKR Apple Bidco, LLC",transportation infrastructure,First Lien Term Loan,SOFR(M),2.50,6.17,NaN,9/23/2031,330359,330359,330927,568,debt investments(a)
465,"OpenMarket, Inc. (Infobip) (United Kingdom)",wireless telecommunication services,First Lien Term Loan,SOFR(Q),5.50,9.20,0.75,6/11/2029,43689841,43166729,42992725,-174004,debt investments(a)
466,Streamland Media Holdings LLC,media,Common Stock,NaN,NaN,NaN,NaN,NaN,12363,645215,0,-645215,equity securities
467,"48forty Intermediate Holdings, Inc.",paper and forest products,Class A-1 Common Units,NaN,NaN,NaN,NaN,NaN,285,0,0,0,equity securities


### Dumping to GS

In [31]:
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]

creds = Credentials.from_service_account_file(
    "service_account.json",
    scopes=SCOPES
)

client = gspread.authorize(creds)

In [32]:
df_map = {
    "Summary": df_summary,
    "All Investments": df_investment_formatted,
    "Instrument Level Analysis": df_instrument_level_analysis,
    "Industry Level Analysis": df_industry_level_analysis,
    "Investment Type level Analysis": df_investmentsType_level_analysis
}

In [33]:
spreadsheet = client.open("BDEBT-10Q-20260331")

In [34]:
for tab_name, df_analysis in df_map.items():
    try:
        worksheet = spreadsheet.worksheet(tab_name)
        worksheet.clear()
    except gspread.WorksheetNotFound:
        worksheet = spreadsheet.add_worksheet(
            title=tab_name,
            rows=max(len(df_analysis) + 10, 100),
            cols=max(len(df_analysis.columns) + 10, 20)
        )

    display(df_analysis)
    set_with_dataframe(worksheet, df_analysis)
    
    header_fmt = CellFormat(
        backgroundColor=Color(0.85, 0.90, 1.0),
        textFormat=TextFormat(bold=True)
    )

    format_cell_range(
        worksheet,
        "1:1",
        header_fmt
    )

    worksheet.freeze(rows=1)
    worksheet.columns_auto_resize(0, len(df_analysis.columns))

,a,b
0,Total Investment Count,469
1,Total Cost,2391096696
2,Total FV,2354502240
3,P&L,-36594456


,Issuer,Industry,Instrument,Ref,Spread,Total Coupon,Floor,Maturity,Principal,Cost,Fair Value,P&L,Investment Type
0,"Arcline FM Holdings, LLC (Fairbanks Morse)",aerospace and defense,First Lien Term Loan,SOFR(Q),2.75,6.45,0.75,6/23/2030,2875972,2881744,2883967,2223,debt investments(a)
1,Bleriot US Bidco Inc.,aerospace and defense,First Lien Term Loan,SOFR(Q),2.50,6.20,NaN,10/30/2026,3289161,3289123,3297960,8837,debt investments(a)
2,Cobham Ultra US Co-Borrower LLC (Ultra Electro...,aerospace and defense,First Lien Term Loan,SOFR(S),4.18,7.79,NaN,8/4/2029,2416311,2417686,2425070,7384,debt investments(a)
3,Chromalloy Holdings LLC,aerospace and defense,First Lien Term Loan,SOFR(Q),3.25,6.91,NaN,3/31/2031,3792109,3806939,3799523,-7416,debt investments(a)
4,"Engineering Research Holding LLC (Astrion, Inc.)",aerospace and defense,First Lien Term Loan,SOFR(Q),5.00,8.70,1.00,8/29/2031,22749759,22484214,18048180,-4436034,debt investments(a)
...,...,...,...,...,...,...,...,...,...,...,...,...,...
464,"KKR Apple Bidco, LLC",transportation infrastructure,First Lien Term Loan,SOFR(M),2.50,6.17,NaN,9/23/2031,330359,330359,330927,568,debt investments(a)
465,"OpenMarket, Inc. (Infobip) (United Kingdom)",wireless telecommunication services,First Lien Term Loan,SOFR(Q),5.50,9.20,0.75,6/11/2029,43689841,43166729,42992725,-174004,debt investments(a)
466,Streamland Media Holdings LLC,media,Common Stock,NaN,NaN,NaN,NaN,NaN,12363,645215,0,-645215,equity securities
467,"48forty Intermediate Holdings, Inc.",paper and forest products,Class A-1 Common Units,NaN,NaN,NaN,NaN,NaN,285,0,0,0,equity securities


,Instrument,Total Investments Count,Total Instrument Cost,Total Instrument FV,Total Instrument P&L,Avg Profit/Instrument,Max Profit Instrument,Max Loss Instrument,Profitable Instrument Count,Profitable %,Breakeven Instrument Count,Breakeven %,Loss Instrument Count,Loss %
19,First Lien Term Loan,283,2067646155,2036917884,-30728271,-108580.46,641498,-4436034,90,0.32,1,0.0,192,0.68
3,First Lien Delayed Draw Term Loan,88,219968696,218950050,-1018646,-11575.52,437255,-762261,37,0.42,2,0.02,49,0.56
11,First Lien Incremental Term Loan,4,20238865,20171821,-67044,-16761.0,16097,-60713,1,0.25,0,0.0,3,0.75
25,First Lien Tranche B-1 Term Loan,1,16677136,16034226,-642910,-642910.0,-642910,-642910,0,0.0,0,0.0,1,1.0
20,First Lien Term Loan (4.5% Exit Fee),1,9391825,9362401,-29424,-29424.0,-29424,-29424,0,0.0,0,0.0,1,1.0
15,First Lien Revolver,64,8519140,7719958,-799182,-12487.22,64331,-381131,23,0.36,2,0.03,39,0.61
24,First Lien Tranche B Term Loan,1,7752306,7499226,-253080,-253080.0,-253080,-253080,0,0.0,0,0.0,1,1.0
12,First Lien Initial Term Loan,1,6640840,6500539,-140301,-140301.0,-140301,-140301,0,0.0,0,0.0,1,1.0
8,First Lien Delayed Draw Term Loan E,1,5783200,5765984,-17216,-17216.0,-17216,-17216,0,0.0,0,0.0,1,1.0
22,First Lien Term Loan B,3,5268451,5183043,-85408,-28469.33,6688,-84798,1,0.33,0,0.0,2,0.67


,Industry,Total Investment Count,Total Investment Cost,Total Investment FV,Total Investment P&L,Avg Profit/Investment,Max Profit Investment,Max Loss Investment,Profitable Investment Count,Profitable %,Breakeven Investment Count,Breakeven %,Loss Investment Count,Loss %
41,software,81,438259932,429222605,-9037327,-111571.94,317772,-793944,12,0.15,0,0.0,69,0.85
38,professional services,41,242852780,233597811,-9254969,-225730.95,58172,-4246664,10,0.24,0,0.0,31,0.76
9,construction and engineering,35,181604353,184052798,2448445,69955.57,641498,-595929,29,0.83,0,0.0,6,0.17
14,diversified financial services,33,180410108,180736974,326866,9905.03,437255,-105217,8,0.24,1,0.03,24,0.73
5,capital markets,19,129052012,128983687,-68325,-3596.05,467649,-193246,6,0.32,0,0.0,13,0.68
26,insurance,25,116242161,116101374,-140787,-5631.48,71608,-126988,11,0.44,2,0.08,12,0.48
21,health care technology,13,93786474,93015506,-770968,-59305.23,458746,-483468,3,0.23,0,0.0,10,0.77
22,healthcare providers and services,20,89182254,89083537,-98717,-4935.85,184821,-210917,6,0.3,0,0.0,14,0.7
0,aerospace and defense,18,85529495,81191723,-4337772,-240987.33,47397,-4436034,11,0.61,0,0.0,7,0.39
23,"hotels, restaurants and leisure",15,74576620,74378756,-197864,-13190.93,142403,-117809,5,0.33,0,0.0,10,0.67


,Investment Type,Total Investment Count,Total Investment Type Cost,Total Investment Type FV,Total Investment Type P&L,Avg Profit/Investment Type,Max Profit Investment Type,Max Loss Investment Type,Profitable Investment Type Count,Profitable %,Breakeven Investment Type Count,Breakeven %,Loss Investment Type Count,Loss %
0,debt investments(a),466,2389364013,2353460431,-35903582,-77046.31,641498,-4436034,156,0.33,8,0.02,302,0.65
1,equity securities,3,1732683,1041809,-690874,-230291.33,0,-645215,0,0.0,1,0.33,2,0.67


In [75]:
for i in df_investment['investment_type'].unique(): print(i)

debt investments(a)
equity securities
